In [0]:
%run ../source_to_bronze/utils

In [0]:
df = spark.read.table("dim_employee")
display(df)

In [0]:
from pyspark.sql.functions import *

salary_df = df.groupBy("department_id") \
    .agg(sum("salary").alias("total_salary")) \
    .orderBy(desc("total_salary"))
display(salary_df)

In [0]:
emp_count_df = df.groupBy("department_id", "country_id").count()
display(emp_count_df)

In [0]:
dept_df = dept_df.toDF("DepartmentID", "DepartmentName")
country_df = country_df.toDF("CountryID", "CountryName")

dept_df = dept_df.toDF("DepartmentID", "DepartmentName")
country_df = country_df.toDF("CountryID", "CountryName")

In [0]:
from pyspark.sql.functions import col

joined_df = df.alias("e") \
    .join(dept_df.alias("d"), col("e.department_id") == col("d.DepartmentID")) \
    .join(country_df.alias("c"), col("e.country_id") == col("c.CountryID")) \
    .select(
        col("d.DepartmentName"),
        col("c.CountryName")
    )

display(joined_df)

In [0]:
avg_age_df = df.groupBy("department_id") \
    .agg(avg("age").alias("avg_age"))
display(avg_age_df)


In [0]:
salary_df = salary_df.withColumn("at_load_date", current_date())
emp_count_df = emp_count_df.withColumn("at_load_date", current_date())
joined_df = joined_df.withColumn("at_load_date", current_date())
avg_age_df = avg_age_df.withColumn("at_load_date", current_date())



In [0]:
gold_path = "/Volumes/assignment/default/employee_vol/gold/employee/"


salary_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save(gold_path + "fact_employee_salary")

emp_count_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save(gold_path + "fact_employee_count")

joined_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save(gold_path + "fact_employee_dept_country")

avg_age_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save(gold_path + "fact_employee_avg_age")